# BDLegalRAG — Stage 3: Chunking Experiment (RQ1)

তিনটা chunking strategy তুলনা করা হবে:
1. **Fixed-size** (naive baseline, 256 token, 50 token overlap)
2. **Recursive character splitter** (sentence/paragraph-boundary সম্মান করে)
3. **Section-aware** (আপনার মূল contribution — প্রতিটা Section আস্ত এক chunk)

প্রতিটার জন্য `BAAI/bge-m3` embedding দিয়ে retrieval index বানিয়ে, `gold_qa_set_v1.csv` দিয়ে **Recall@k, Precision@k, MRR** মাপা হবে।

**নোট:** এই Stage-এ embedding model হিসেবে শুধু BGE-M3 ব্যবহার হচ্ছে (chunking-এর প্রভাব আলাদাভাবে দেখার জন্য)। BGE-M3 vs BanglaSBERT তুলনা হবে পরের ধাপে, **Stage 4 (Embedding Benchmark)**-এ।


## ধাপ ০: Setup

In [ ]:
import sys, os, json, re

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/BDLegalRAG"
else:
    BASE_DIR = "/kaggle/working/BDLegalRAG"

PROCESSED_DIR = os.path.join(BASE_DIR, "processed_data")
QA_DIR = os.path.join(BASE_DIR, "qa_data")
CHUNK_DIR = os.path.join(BASE_DIR, "chunking_experiment")
os.makedirs(CHUNK_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/BDLegalRAG


In [ ]:
!pip install sentence-transformers faiss-cpu pandas --quiet
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

print("Libraries ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.3 MB/s eta 0:00:00
Libraries ready.


In [ ]:
with open(os.path.join(PROCESSED_DIR, "bdlegalrag_corpus_v1.json"), encoding="utf-8") as f:
    corpus = json.load(f)

gold_df = pd.read_csv(os.path.join(QA_DIR, "gold_qa_set_v1.csv"))

print("Total acts:", len(corpus["acts"]))
print("Total gold QA pairs:", len(gold_df))


Total acts: 49
Total gold QA pairs: 180


In [ ]:
import json, re, os

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

def parse_section_number_direct(text):
    if not isinstance(text, str):
        return None
    text = text.strip()
    m = re.match(r"^(?:\d+\[)?\s*([০-৯]+[কখগঘঙ]?)[।৷]", text)
    if not m:
        return None
    bn_num = m.group(1)
    digits = "".join(ch for ch in bn_num if ch in BN_DIGITS)
    suffix = "".join(ch for ch in bn_num if ch not in BN_DIGITS)
    return digits.translate(BN_TO_EN) + suffix

CORPUS_PATH = os.path.join(BASE_DIR, "processed_data", "bdlegalrag_corpus_v1.json")
with open(CORPUS_PATH, encoding="utf-8") as f:
    corpus = json.load(f)

fixed_count = 0
for act in corpus["acts"]:
    for sec in act["sections"]:
        if not sec.get("section_number"):
            new_num = parse_section_number_direct(sec["section_text"])
            if new_num:
                sec["section_number"] = new_num
                fixed_count += 1

print(f"Fixed {fixed_count} sections directly in the corpus JSON.")

with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print("Corpus JSON overwritten with corrected section numbers.")

Fixed 0 sections directly in the corpus JSON.
Corpus JSON overwritten with corrected section numbers.


## ধাপ ১: তিনটা Chunking Strategy Implement করুন

প্রতিটা strategy একই input (corpus-এর সব section) থেকে ভিন্নভাবে chunk বানাবে। প্রতিটা chunk-এ metadata থাকবে (`act_title`, `section_number`) — যাতে retrieval-এর পর "সঠিক Section পাওয়া গেছে কিনা" যাচাই করা যায়।


In [ ]:
def word_count(text):
    return len(text.split())


def chunk_fixed_size(sections, chunk_size=256, overlap=50):
    """Naive baseline: split each section's text into fixed-size word windows,
    ignoring sentence/paragraph boundaries entirely."""
    chunks = []
    for sec in sections:
        words = sec["section_text"].split()
        if len(words) <= chunk_size:
            chunks.append({**sec, "chunk_text": sec["section_text"]})
            continue
        start = 0
        while start < len(words):
            end = start + chunk_size
            chunk_text = " ".join(words[start:end])
            chunks.append({**sec, "chunk_text": chunk_text})
            start += chunk_size - overlap
    return chunks


def chunk_recursive(sections, max_chunk_size=256):
    """Recursive splitter: try to split on paragraph, then sentence (।/৷ danda),
    then word boundary — only descending to a finer separator when a piece is
    still too large. Keeps natural language boundaries where fixed-size doesn't."""
    separators = ["\n\n", "।", "৷", " "]

    def split_text(text, seps):
        if word_count(text) <= max_chunk_size or not seps:
            return [text]
        sep = seps[0]
        parts = [p for p in text.split(sep) if p.strip()]
        if len(parts) <= 1:
            return split_text(text, seps[1:])
        result = []
        buffer = ""
        for part in parts:
            candidate = (buffer + sep + part) if buffer else part
            if word_count(candidate) <= max_chunk_size:
                buffer = candidate
            else:
                if buffer:
                    result.extend(split_text(buffer, seps[1:]))
                buffer = part
        if buffer:
            result.extend(split_text(buffer, seps[1:]))
        return result

    chunks = []
    for sec in sections:
        pieces = split_text(sec["section_text"], separators)
        for piece in pieces:
            chunks.append({**sec, "chunk_text": piece})
    return chunks


def chunk_section_aware(sections):
    """Your core contribution: one chunk per Section, exactly as BLAD/the corpus
    already structures it — no further splitting, hierarchy fully preserved."""
    return [{**sec, "chunk_text": sec["section_text"]} for sec in sections]


# corpus থেকে flat section list বানানো (chunking strategy-গুলোর input)
all_sections = []
for act in corpus["acts"]:
    for sec in act["sections"]:
        all_sections.append({
            "act_title": act["act_title"],
            "act_no": act.get("act_no"),
            "section_number": sec.get("section_number"),
            "section_text": sec["section_text"],
        })

print(f"Total sections available: {len(all_sections)}")


Total sections available: 1575


In [ ]:
strategies = {
    "fixed_size": chunk_fixed_size(all_sections),
    "recursive": chunk_recursive(all_sections),
    "section_aware": chunk_section_aware(all_sections),
}

for name, chunks in strategies.items():
    lengths = [word_count(c["chunk_text"]) for c in chunks]
    print(f"{name}: {len(chunks)} chunks | avg words: {sum(lengths)/len(lengths):.1f} | "
          f"min: {min(lengths)} | max: {max(lengths)}")


fixed_size: 1764 chunks | avg words: 98.9 | min: 1 | max: 256
recursive: 1782 chunks | avg words: 92.9 | min: 1 | max: 256
section_aware: 1575 chunks | avg words: 105.0 | min: 1 | max: 2830


## ধাপ ২: BGE-M3 দিয়ে Embed করে FAISS Index বানান

প্রতিটা strategy-র chunk গুলো আলাদাভাবে embed হবে, আলাদা FAISS index-এ যাবে।

**সময়ের হিসাব:** BGE-M3 একটা 567M-parameter মডেল, CPU-তে চলবে কিন্তু ধীরে। এই corpus-এর আকারে (~kilobytes of chunks) মোটামুটি ১৫-৩০ মিনিট লাগতে পারে সবগুলো strategy মিলিয়ে — ধৈর্য ধরুন, একবারই করতে হবে।


In [ ]:
print("Loading BAAI/bge-m3 (first run downloads ~4.5GB, subsequent runs are cached)...")
embedder = SentenceTransformer("BAAI/bge-m3")
print("Model loaded. Embedding dimension:", embedder.get_sentence_embedding_dimension())


Loading BAAI/bge-m3 (first run downloads ~4.5GB, subsequent runs are cached)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (10 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 1024


/tmp/ipykernel_1188/3073105208.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", embedder.get_sentence_embedding_dimension())


In [ ]:
import pickle

EMBEDDING_CACHE_PATH = os.path.join(CHUNK_DIR, "embeddings_cache.pkl")

indices = {}       # strategy_name -> faiss index
chunk_lookup = {}   # strategy_name -> list of chunk dicts (same order as index)

if os.path.exists(EMBEDDING_CACHE_PATH):
    # Session reconnect হলেও আবার embed করতে হবে না — Drive-এ cache থেকে সরাসরি load
    print("Cache পাওয়া গেছে, embedding আবার করা হচ্ছে না — সরাসরি load করা হচ্ছে...")
    with open(EMBEDDING_CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    for name, (embeddings, chunks) in cache.items():
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        indices[name] = index
        chunk_lookup[name] = chunks
        print(f"  -> '{name}' loaded from cache: {index.ntotal} vectors")
else:
    print("Cache নেই, প্রথমবার embed করা হচ্ছে (এই ধাপটাই সবচেয়ে সময় নেবে)...")
    cache_to_save = {}
    for name, chunks in strategies.items():
        print(f"Embedding '{name}' ({len(chunks)} chunks)...")
        texts = [c["chunk_text"] for c in chunks]
        embeddings = embedder.encode(
            texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True
        )
        embeddings = np.asarray(embeddings, dtype="float32")

        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        indices[name] = index
        chunk_lookup[name] = chunks
        cache_to_save[name] = (embeddings, chunks)
        print(f"  -> indexed {index.ntotal} vectors\n")

    with open(EMBEDDING_CACHE_PATH, "wb") as f:
        pickle.dump(cache_to_save, f)
    print(f"Cache saved to {EMBEDDING_CACHE_PATH} — পরের বার reconnect হলে এই ধাপ সেকেন্ডে শেষ হবে।")

print("\nAll strategies ready (indices + chunk_lookup populated).")


Cache পাওয়া গেছে, embedding আবার করা হচ্ছে না — সরাসরি load করা হচ্ছে...
  -> 'fixed_size' loaded from cache: 1764 vectors
  -> 'recursive' loaded from cache: 1782 vectors
  -> 'section_aware' loaded from cache: 1575 vectors

All strategies ready (indices + chunk_lookup populated).


## ধাপ ৩: প্রতিটা Gold QA দিয়ে Retrieval Evaluation চালান

প্রতিটা প্রশ্নের জন্য: প্রশ্নটা embed করে top-k chunk retrieve করা হবে, তারপর দেখা হবে সেই top-k-এর মধ্যে **সঠিক Section (gold-এ যেটা আছে)** আছে কিনা।

- **Recall@k**: সঠিক section top-k-এ পাওয়া গেছে কিনা (1/0), সব প্রশ্নের গড়
- **Precision@k**: top-k-এর মধ্যে কয়টা আসলেই relevant (এখানে প্রতি প্রশ্নে ১টাই gold section থাকায়, hit/k)
- **MRR (Mean Reciprocal Rank)**: সঠিক section কোন rank-এ পাওয়া গেছে তার gড় বিপরীত মান — এটা extra, but standard practice for single-answer retrieval


In [ ]:
BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

def normalize_section_number(val):
    """তুলনার আগে যেকোনো section_number-কে একই format-এ আনা —
    বাংলা সংখ্যা থাকলে ইংরেজিতে বদলে, সব lowercase/strip করে।"""
    if val is None:
        return None
    s = str(val).strip().translate(BN_TO_EN)
    return s

def is_match(chunk, gold_row):
    same_act = str(chunk["act_title"]).strip() == str(gold_row["act_title"]).strip()
    same_section = normalize_section_number(chunk["section_number"]) == normalize_section_number(gold_row["section_number"])
    return same_act and same_section

# def is_match(chunk, gold_row):
#     """একটা retrieved chunk কি gold QA-র সঠিক Section-এর সাথে মেলে।"""
#     same_act = str(chunk["act_title"]).strip() == str(gold_row["act_title"]).strip()
#     same_section = str(chunk["section_number"]).strip() == str(gold_row["section_number"]).strip()
#     return same_act and same_section


def evaluate_strategy(name, k_values=(5, 10, 20)):
    index = indices[name]
    chunks = chunk_lookup[name]

    results = {k: {"recall_hits": 0, "precision_sum": 0.0} for k in k_values}
    reciprocal_ranks = []
    max_k = max(k_values)

    query_embeddings = embedder.encode(
        gold_df["question"].tolist(), batch_size=16, show_progress_bar=True,
        normalize_embeddings=True
    )
    query_embeddings = np.asarray(query_embeddings, dtype="float32")

    _, all_indices = index.search(query_embeddings, max_k)

    for qi, gold_row in gold_df.reset_index(drop=True).iterrows():
        retrieved_idx = all_indices[qi]
        retrieved_chunks = [chunks[idx] for idx in retrieved_idx]

        # Reciprocal rank
        rr = 0.0
        for rank, ch in enumerate(retrieved_chunks, start=1):
            if is_match(ch, gold_row):
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

        for k in k_values:
            top_k = retrieved_chunks[:k]
            hit = any(is_match(ch, gold_row) for ch in top_k)
            results[k]["recall_hits"] += int(hit)
            results[k]["precision_sum"] += (1 if hit else 0) / k  # single relevant item per query

    n = len(gold_df)
    summary = {"strategy": name, "MRR": sum(reciprocal_ranks) / n}
    for k in k_values:
        summary[f"Recall@{k}"] = results[k]["recall_hits"] / n
        summary[f"Precision@{k}"] = results[k]["precision_sum"] / n
    return summary


all_results = []
for name in strategies:
    print(f"Evaluating strategy: {name}")
    summary = evaluate_strategy(name)
    all_results.append(summary)
    print(summary, "\n")


Evaluating strategy: fixed_size


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

{'strategy': 'fixed_size', 'MRR': 0.7112606397777966, 'Recall@5': 0.8166666666666667, 'Precision@5': 0.16333333333333294, 'Recall@10': 0.8777777777777778, 'Precision@10': 0.08777777777777755, 'Recall@20': 0.9222222222222223, 'Precision@20': 0.04611111111111102} 

Evaluating strategy: recursive


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

{'strategy': 'recursive', 'MRR': 0.7143708089296326, 'Recall@5': 0.8277777777777777, 'Precision@5': 0.16555555555555515, 'Recall@10': 0.8722222222222222, 'Precision@10': 0.087222222222222, 'Recall@20': 0.9277777777777778, 'Precision@20': 0.0463888888888888} 

Evaluating strategy: section_aware


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

{'strategy': 'section_aware', 'MRR': 0.7117894417101073, 'Recall@5': 0.8222222222222222, 'Precision@5': 0.16444444444444403, 'Recall@10': 0.8777777777777778, 'Precision@10': 0.08777777777777755, 'Recall@20': 0.9277777777777778, 'Precision@20': 0.0463888888888888} 



## ধাপ ৪: ফলাফল তুলনা টেবিল — এটাই আপনার Paper-এর মূল RQ1 Result

In [ ]:
results_df = pd.DataFrame(all_results).set_index("strategy")
results_df = results_df[["MRR", "Recall@5", "Precision@5", "Recall@10", "Precision@10", "Recall@20", "Precision@20"]]
results_df = results_df.round(4)

print(results_df.to_string())

results_path = os.path.join(CHUNK_DIR, "chunking_comparison_results.csv")
results_df.to_csv(results_path)
print(f"\nSaved to: {results_path}")


                  MRR  Recall@5  Precision@5  Recall@10  Precision@10  Recall@20  Precision@20
strategy                                                                                      
fixed_size     0.7113    0.8167       0.1633     0.8778        0.0878     0.9222        0.0461
recursive      0.7144    0.8278       0.1656     0.8722        0.0872     0.9278        0.0464
section_aware  0.7118    0.8222       0.1644     0.8778        0.0878     0.9278        0.0464

Saved to: /content/drive/MyDrive/BDLegalRAG/chunking_experiment/chunking_comparison_results.csv


## [ANALYSIS] Failure Analysis — কোন প্রশ্নগুলো এখনো Retrieval করতে ব্যর্থ

Recall@20 ~০.৮৯ মানে ~১১% প্রশ্ন top-20-এও সঠিক Section পায়নি। উন্নতির আগে এই প্রশ্নগুলো নিজে চোখে দেখে নিন — প্যাটার্ন খুঁজে বের করাই পরবর্তী fix ঠিক করার সবচেয়ে ভালো উপায়।


In [ ]:
# section_aware strategy (বা আপনার পছন্দের strategy) দিয়ে failure cases বের করা
BEST_STRATEGY = "recursive"  # আপনার ফলাফল অনুযায়ী বদলে নিন
K_FOR_ANALYSIS = 20

index = indices[BEST_STRATEGY]
chunks = chunk_lookup[BEST_STRATEGY]

query_embeddings = embedder.encode(gold_df["question"].tolist(), normalize_embeddings=True)
_, all_indices = index.search(np.asarray(query_embeddings, dtype="float32"), K_FOR_ANALYSIS)

failed_queries = []
for qi, gold_row in gold_df.reset_index(drop=True).iterrows():
    retrieved_chunks = [chunks[idx] for idx in all_indices[qi]]
    hit = any(is_match(ch, gold_row) for ch in retrieved_chunks)
    if not hit:
        failed_queries.append({
            "qa_id": gold_row.get("qa_id", qi),
            "question": gold_row["question"],
            "gold_act": gold_row["act_title"],
            "gold_section": gold_row["section_number"],
            "top1_retrieved_act": retrieved_chunks[0]["act_title"],
            "top1_retrieved_section": retrieved_chunks[0]["section_number"],
        })

failed_df = pd.DataFrame(failed_queries)
print(f"Total failed queries (miss even at k={K_FOR_ANALYSIS}): {len(failed_df)} out of {len(gold_df)}")
print()
print(failed_df.to_string())

failed_path = os.path.join(CHUNK_DIR, f"failed_queries_{BEST_STRATEGY}.csv")
failed_df.to_csv(failed_path, index=False, encoding="utf-8-sig")
print(f"\nSaved to: {failed_path} — পড়ে দেখুন এদের মধ্যে কোনো common pattern আছে কিনা")


Total failed queries (miss even at k=20): 13 out of 180

      qa_id                                                                                                                                                        question                                            gold_act gold_section                                  top1_retrieved_act top1_retrieved_section
0   qa_0049                                                                        নারী ও শিশু নির্যাতন দমন আইনের সংশোধনে জরিমানা বা অর্থের পরিমাণগুলো কীভাবে বাড়ানো হয়েছে?    নারী ও শিশু নির্যাতন দমন (সংশোধন) অধ্যাদেশ, ২০২৫            3    নারী ও শিশু নির্যাতন দমন (সংশোধন) অধ্যাদেশ, ২০২৫                      4
1   qa_0056  শিশু দিবাযত্ন কেন্দ্র আইন, ২০২১ তো বাংলায় আছে, সরকার কি এটার ইংরেজি কোনো অনুবাদ প্রকাশ করতে পারবে? আর বাংলা ও ইংরেজির মধ্যে কোনো মিল না থাকলে কোনটা মানা হবে?                     শিশু দিবাযত্ন কেন্দ্র আইন, ২০২১           34                            ভূমি উন্নয়ন কর আইন, ২০২৩                     23
2   qa_

## পরের ধাপ

এখানে যে strategy সবচেয়ে ভালো Recall@k/MRR দেখায়, সেটাই আপনার **RQ1-এর মূল finding** — paper-এ এই টেবিলটাই আপনার প্রথম result table হবে।

**Stage 4 (Embedding Benchmark)**-এ এই একই evaluation-টা BanglaSBERT দিয়ে আবার চালিয়ে BGE-M3-র সাথে তুলনা করা হবে — সেরা chunking strategy + সেরা embedding মডেল, এই দুটো মিলিয়ে আপনার Stage 5 (Hybrid Retrieval)-এর ভিত্তি তৈরি হবে।

চাইলে এখনই Stage 4-এর notebook বানিয়ে দেব?


final done